# 特徵融合

[完整圖文教材](https://birdhackor.github.io/learn_to_yolo/lessons/11-fusion/)

先執行下一格的環境格，再閱讀、執行最後的完整實驗。各節互相獨立，不需要先跑其他節。

最後一格存著這段程式在 CPU 上實際執行的輸出；執行環境與其他實驗紀錄見網站的〈驗證範圍〉。範例不用預訓練權重，也不需要下載資料。

In [ ]:
# 全新 runtime 可由此格開始。只取得小型程式，不自動下載 LFS 資料。
import importlib.metadata
import os
from pathlib import Path
import subprocess
import sys

REF = 'lessons-v0.4.1'
base = Path("/content") if Path("/content").is_dir() else Path.cwd()
repository = base / ("learn_to_yolo_" + REF.replace(".", "_").replace("-", "_"))
if not repository.exists():
    environment = os.environ.copy()
    environment["GIT_LFS_SKIP_SMUDGE"] = "1"
    subprocess.run(["git", "clone", "--depth", "1", "--branch", REF,
                    "https://github.com/birdhackor/learn_to_yolo.git", str(repository)],
                   env=environment, check=True)
actual_ref = subprocess.check_output(
    ["git", "-C", str(repository), "describe", "--tags", "--exact-match"], text=True
).strip()
if actual_ref != REF:
    raise RuntimeError("程式版本不同，請使用新的 runtime 或移除舊 clone 後重試。")

# 各節實驗都在 CPU 上執行，執行紀錄用的是 PyTorch 2.9.1。已經是 2.9.1 就沿用（CPU 或 CUDA 版都可以）；
# 其他版本會改裝成 2.9.1 的 CPU 版，之後這個工作階段就用不到 GPU，但各節實驗本來就不需要。
try:
    torch_version = importlib.metadata.version("torch").split("+")[0]
except importlib.metadata.PackageNotFoundError:
    torch_version = None
if torch_version != "2.9.1":
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "torch==2.9.1",
                    "--index-url", "https://download.pytorch.org/whl/cpu"], check=True)
    if "torch" in sys.modules:
        raise RuntimeError("PyTorch 已更新：請從選單重新啟動工作階段，再由第一格執行。")
subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                *['numpy==2.3.5', 'pillow==12.0.0', 'matplotlib==3.10.7']], check=True)
os.chdir(repository)
sys.path.insert(0, str(repository))
print("固定教材版本：", REF)
print("目前目錄：", repository)


## 本節可修改的完整實驗

先預測結果，再執行；確認輸出後，試做網頁的自主練習。

In [1]:
import torch
from torch import nn
import torch.nn.functional as F


class Fusion(nn.Module):
    def __init__(self):
        super().__init__()
        self.reduce = nn.Conv2d(16,8,1)
        self.mix = nn.Conv2d(16,8,3,padding=1)

    def forward(self,shallow,deep,inspect=False):
        reduced = self.reduce(deep)
        up = F.interpolate(reduced,size=shallow.shape[-2:],mode='nearest')
        concat = torch.cat([shallow,up],dim=1)
        output = self.mix(concat)
        return (output,reduced,up,concat) if inspect else output


def main():
    torch.manual_seed(7)
    torch.set_num_threads(2)
    shallow = torch.randn(1,8,8,8,requires_grad=True)
    deep = torch.randn(1,16,4,4,requires_grad=True)
    model = Fusion()
    optimizer = torch.optim.SGD(model.parameters(),lr=.01)
    before = model.mix.weight.detach().clone()
    # The model's only forward pass; inspect=True also returns the intermediate tensors measured below.
    output,reduced,up,concat = model(shallow,deep,inspect=True)
    assert output.shape == (1,8,8,8)
    loss = output.square().mean()
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    assert torch.isfinite(shallow.grad).all() and torch.isfinite(deep.grad).all()
    assert all(p.grad is not None and torch.isfinite(p.grad).all() and p.grad.abs().sum() > 0
               for p in model.parameters())
    assert shallow.grad.abs().sum() > 0 and deep.grad.abs().sum() > 0
    optimizer.step()
    assert not torch.equal(before,model.mix.weight.detach())
    # Different channels can legally concatenate; only spatial axes must match.
    unreduced = F.interpolate(deep.detach(),size=(8,8),mode='nearest')
    joined = torch.cat([shallow.detach(),unreduced],dim=1)
    assert joined.shape == (1,24,8,8)
    assert nn.Conv2d(24,8,3,padding=1)(joined).shape == (1,8,8,8)
    # After reduce, forward()'s up has shallow's 8 channels, so adding them would also be legal.
    assert concat.shape == (1,16,8,8)
    assert (shallow.detach()+up.detach()).shape == (1,8,8,8)
    # A fully hand-checkable nearest-neighbor interpolation example.
    toy = torch.tensor([[[[1.,2.],[3.,4.]]]],requires_grad=True)
    enlarged = F.interpolate(toy,size=(4,4),mode='nearest')
    expected = torch.tensor([[[[1.,1.,2.,2.],[1.,1.,2.,2.],
                               [3.,3.,4.,4.],[3.,3.,4.,4.]]]])
    assert torch.equal(enlarged,expected)
    enlarged.sum().backward()
    assert torch.equal(toy.grad,torch.full_like(toy,4))
    print('nearest example',enlarged[0,0].tolist(),'source gradient',toy.grad[0,0].tolist())
    print('shapes shallow',tuple(shallow.shape),'deep',tuple(deep.shape),'-> reduce',tuple(reduced.shape),
          '-> nearest',tuple(up.shape),'-> concat',tuple(concat.shape),'-> mix',tuple(output.shape))
    print('parameters',sum(p.numel() for p in model.parameters()),'both branches backward and one step')


if __name__ == '__main__':
    main()


nearest example [[1.0, 1.0, 2.0, 2.0], [1.0, 1.0, 2.0, 2.0], [3.0, 3.0, 4.0, 4.0], [3.0, 3.0, 4.0, 4.0]] source gradient [[4.0, 4.0], [4.0, 4.0]]
shapes shallow (1, 8, 8, 8) deep (1, 16, 4, 4) -> reduce (1, 8, 4, 4) -> nearest (1, 8, 8, 8) -> concat (1, 16, 8, 8) -> mix (1, 8, 8, 8)
parameters 1296 both branches backward and one step
